Clean & Aggregate EV Data

In [25]:
import pandas as pd

# Load raw data
ev_df = pd.read_csv("../data/Electric_Vehicle_Charging_Station_Data.csv")

# Preview
print(ev_df.shape)
print(ev_df.columns)
ev_df.head()

(148136, 17)
Index(['Station_Name', 'Address', 'City', 'State_Province', 'Zip_Postal_Code',
       'Start_Date___Time', 'Start_Time_Zone', 'End_Date___Time',
       'End_Time_Zone', 'Total_Duration__hh_mm_ss_',
       'Charging_Time__hh_mm_ss_', 'Energy__kWh_', 'GHG_Savings__kg_',
       'Gasoline_Savings__gallons_', 'Port_Type', 'ObjectID', 'ObjectId2'],
      dtype='object')


,Station_Name,Address,City,State_Province,Zip_Postal_Code,Start_Date___Time,Start_Time_Zone,End_Date___Time,End_Time_Zone,Total_Duration__hh_mm_ss_,Charging_Time__hh_mm_ss_,Energy__kWh_,GHG_Savings__kg_,Gasoline_Savings__gallons_,Port_Type,ObjectID,ObjectId2
0,BOULDER / JUNCTION ST1,2280 Junction Pl,Boulder,Colorado,80301,01/01/2018 17:49,MDT,01/01/2018 19:52,MDT,02:03:02,02:02:44,6.504,2.732,0.816,Level 2,0,1
1,BOULDER / JUNCTION ST1,2280 Junction Pl,Boulder,Colorado,80301,01/02/2018 08:52,MDT,01/02/2018 09:16,MDT,00:24:34,00:24:19,2.481,1.042,0.311,Level 2,1,2
2,BOULDER / JUNCTION ST1,2280 Junction Pl,Boulder,Colorado,80301,01/02/2018 21:11,MDT,01/03/2018 06:23,MDT,09:12:21,03:40:52,15.046,6.319,1.888,Level 2,2,3
3,BOULDER / ALPINE ST1,1275 Alpine Ave,Boulder,Colorado,80304,01/03/2018 09:19,MDT,01/03/2018 11:14,MDT,01:54:51,01:54:29,6.947,2.918,0.872,Level 2,3,4
4,BOULDER / BASELINE ST1,900 Baseline Rd,Boulder,Colorado,80302,01/03/2018 14:13,MDT,01/03/2018 14:30,MDT,00:16:58,00:16:44,1.800,0.756,0.226,Level 2,4,5


In [26]:
# Drop ObjectID and ObjectId2 columns if they exist
ev_df.drop(columns=['ObjectID', 'ObjectId2'], inplace=True, errors='ignore')

In [30]:
# 2. Drop duplicate records
ev_df.drop_duplicates(inplace=True)
ev_df.shape
ev_df.head

<bound method NDFrame.head of                      Station_Name            Address     City State_Province  \
0          BOULDER / JUNCTION ST1   2280 Junction Pl  Boulder       Colorado   
1          BOULDER / JUNCTION ST1   2280 Junction Pl  Boulder       Colorado   
2          BOULDER / JUNCTION ST1   2280 Junction Pl  Boulder       Colorado   
3            BOULDER / ALPINE ST1    1275 Alpine Ave  Boulder       Colorado   
4          BOULDER / BASELINE ST1    900 Baseline Rd  Boulder       Colorado   
...                           ...                ...      ...            ...   
148131  BOULDER / N BOULDER REC 1      3172 Broadway  Boulder       Colorado   
148132  BOULDER / CARPENTER PARK1       1505 30th St  Boulder       Colorado   
148133  BOULDER / CARPENTER PARK1       1505 30th St  Boulder       Colorado   
148134   BOULDER / REC CENTER ST2  1360 Gillaspie Dr  Boulder       Colorado   
148135   BOULDER / FACILITIES ST1   1745 14th street  Boulder       Colorado   

        Z

In [31]:
# Convert Start_Date___Time to datetime format (auto-infer format)
ev_df['Start_Date___Time'] = pd.to_datetime(ev_df['Start_Date___Time'], dayfirst=True, errors='coerce')

# Create a new 'date' column (keeping only the date part)
ev_df['date'] = ev_df['Start_Date___Time'].dt.date

# Optional: convert 'date' to datetime type if needed later
ev_df['date'] = pd.to_datetime(ev_df['date'])

# Preview
print(ev_df[['Start_Date___Time', 'date']].head())

    Start_Date___Time       date
0 2018-01-01 17:49:00 2018-01-01
1 2018-02-01 08:52:00 2018-02-01
2 2018-02-01 21:11:00 2018-02-01
3 2018-03-01 09:19:00 2018-03-01
4 2018-03-01 14:13:00 2018-03-01


In [33]:
# Group by date and sum energy (daily total energy consumption in kWh)
ev_daily = ev_df.groupby('date')['Energy__kWh_'].sum().reset_index()

# Rename column for clarity
ev_daily.rename(columns={'Energy__kWh_': 'energy'}, inplace=True)

# Sort by date just in case
ev_daily.sort_values('date', inplace=True)

# Reset index (optional, for clean dataframe)
ev_daily.reset_index(drop=True, inplace=True)

# Preview the result
print(ev_daily.head())
print(f"\n✅ Aggregated data shape: {ev_daily.shape}")


        date   energy
0 2018-01-01    6.504
1 2018-01-02  104.703
2 2018-01-03   80.573
3 2018-01-04   41.900
4 2018-01-05  127.434

✅ Aggregated data shape: (911, 2)
